# Dataset Preprocessing
In this notebook we are going to preprocess the slides to generate our sets of patches for training, validation, and testing.

In [ ]:
from pathgen.utils.seeds import set_seed

global_seed = 13371
set_seed(global_seed)

In [ ]:
from pathgen.utils.paths import project_root

experiment_name = "paper"
experiment_root = project_root() / "experiments" / experiment_name

## 1. Audit the dataset
Let's find out how many patches there are in each slide in the Camelyon16 dataset, so we can select from the in a principled way.

In [ ]:
import pathgen.data.datasets.camelyon16 as camelyon16

# create the datasets that we will use
train = camelyon16.training()
test = camelyon16.testing()

print(f'Training slides: {len(train)}')
print(f'Testing slides: {len(test)}')

In [ ]:
from pathgen.preprocess.patching import make_index
from pathgen.preprocess.tissue_detection import TissueDetectorOTSU
from pathgen.preprocess.patching import GridPatchFinder

tissue_detector = TissueDetectorOTSU()
patch_finder = GridPatchFinder(6, 0, 256, 256)

train_index = make_index(train, tissue_detector, patch_finder)

In [ ]:
test_index = make_index(test, tissue_detector, patch_finder)

In [ ]:
def print_totals(index):
    summary = index.summary()
    total_normal = summary['normal'].sum()
    total_tumor = summary['tumor'].sum()
    print(f"Total normal patches: {total_normal}")
    print(f"Total tumor patches: {total_tumor}")
    
print("Train Index")
print_totals(train_index)
print("")
print("Test Index")
print_totals(test_index)

In [ ]:
train_index.save(experiment_root / 'index' / 'train')
test_index.save(experiment_root / 'index' / 'test')

In [ ]:
from pathgen.preprocess.patching import SlidesIndex

loaded_index = SlidesIndex.load(experiment_root / 'index' / 'train')
len(loaded_index)